In [5]:
import sys
import os

project_dir = r"D:/ssbroyden_optimistix"
sys.path.insert(0, project_dir)

from optimistix_wrapper import run_optimization
from ssbrodyen_family import BFGS, SSBFGS, Broyden, SSBroyden, Zoom

from dataclasses import dataclass
import jax
import optax
from jax import random, numpy as jnp
from jax import vmap, jacfwd, jit
from flax import linen as nn
from evojax.util import get_params_format_fn
import os
import time
import numpy as np
from scipy import io
import optimistix as optix

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
#jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

In [6]:
class PINN(nn.Module):
    n_nodes: int
    def setup(self):
        self.layers = [nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       jnp.sin,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu]
        self.last_layer = nn.Dense(1, kernel_init = jax.nn.initializers.he_uniform(), use_bias=False)

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        t, x = inputs[:,0:1], inputs[:,1:2]
        def get_u(t, x):
            f = jnp.hstack([t, x])
            for i, lyr in enumerate(self.layers):
                f = lyr(f)
                if (i == 0):
                    f = 2 *jnp.pi *f
            u = self.last_layer(f)
            return u

        u = get_u(t, x)

        # obtain u_t, u_xx
        def get_u_der(get_u, t, x):
            u_t, u_x = jacfwd(get_u)(t, x), jacfwd(get_u, argnums=1)(t, x)
            u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(t, x)
            return u_t, u_x, u_xx

        f_der_vmap = vmap(get_u_der, in_axes=(None, 0, 0))
        u_t, u_x, u_xx = f_der_vmap(get_u, t, x)
        u_t, u_x, u_xx = u_t[:,:,0], u_x[:,:,0], u_xx[:,:,0,0]

        # obtain BC/IC indices
        ic, bc = (t == t_l), (x == x_l) # (x==x_l2)  # lower x
        pbc = get_u(t, jnp.ones_like(t)*x_u) - u
        nbc = get_u(t, jnp.ones_like(t)*x_u2) - get_u(t, jnp.ones_like(t)*x_l2)

        # PDE: u_t - v2* u_xx + 5*(u**3) - 5*u = 0
        pde = (u_t - v2* u_xx + 5*(u**3) - 5*u)  

        outputs = jnp.hstack([u, u_xx, pde, pbc, nbc, ic, bc])
        return outputs
    

class DNN(nn.Module):
    """DNNs"""
    n_nodes: int
    def setup(self):
        self.layers = [nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       jnp.sin,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu]
        self.last_layer = nn.Dense(1, kernel_init = jax.nn.initializers.he_uniform(), use_bias=False)

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        t, x = inputs[:,0:1], inputs[:,1:2]
        def get_u(t, x):
            f = jnp.hstack([t, x])
            for i, lyr in enumerate(self.layers):
                f = lyr(f)
                if (i == 0):
                    f = 2 *jnp.pi *f
            u = self.last_layer(f)
            return u

        u = get_u(t, x)
        def get_u0_der(get_u, t, x):
            u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(t, x)
            return  u_xx
        
        f_der_vmap = vmap(get_u0_der, in_axes=(None, 0, 0))
        u_xx = f_der_vmap(get_u, t, x)[:,:,0,0]

        outputs = jnp.hstack([u, u_xx])
        return outputs
    
    
class TEST(nn.Module):
    n_nodes: int
    def setup(self):
        self.layers = [nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       jnp.sin,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu,
                       nn.Dense(self.n_nodes, kernel_init = jax.nn.initializers.he_uniform()),
                       nn.silu]
        self.last_layer = nn.Dense(1, kernel_init = jax.nn.initializers.he_uniform(), use_bias=False)

    @nn.compact
    def __call__(self, inputs):
        # split the two variables, probably just by slicing
        t, x = inputs[:,0:1], inputs[:,1:2]
        def get_u(t, x):
            f = jnp.hstack([t, x])
            for i, lyr in enumerate(self.layers):
                f = lyr(f)
                if (i == 0):
                    f = 2 *jnp.pi *f
            u = self.last_layer(f)
            return u

        u = get_u(t, x)
        return u

In [ ]:
def main(ER, ER_xx, weight_ic, weight_bc, max_lr, exponent, max_iter, seed, gpu):
    # 初始化全局变量
    global v2, t_l, t_u, x_l, x_u, x_l2, x_u2
    v2 = 0.0001
    
    # 加载数据
    mat_data = io.loadmat('E:/SCALE-PINN/ac/allen_cahn.mat')
    sim_t = np.repeat(mat_data['t'], 512).reshape(-1, 1)
    sim_x = np.tile(mat_data['x'], 201).reshape(-1, 1)
    sim_u = mat_data['usol'].reshape(-1, 1)
    tt, xx = mat_data['t'], mat_data['x']

    # 设置边界条件
    t_l, t_u, x_l, x_u = np.min(sim_t), np.max(sim_t), np.min(sim_x), np.max(sim_x)
    dt, dx = tt[0, 1] - tt[0, 0], xx[0, 1] - xx[0, 0]
    x_l2, x_u2 = x_l + dx, x_u - dx
    
    # 准备数据集
    data_X, data_Y = np.hstack([sim_t, sim_x]), sim_u
    bic = (data_X[:, 0] == 0) | (data_X[:, 1] == x_l)
    data_X_BIC, data_Y_BIC = data_X[bic], data_Y[bic]

    # 转换为JAX数组
    data_X, data_Y, data_X_BIC, data_Y_BIC = (
        jnp.array(data_X), jnp.array(data_Y), 
        jnp.array(data_X_BIC), jnp.array(data_Y_BIC)
    )

    # 初始化模型
    seed = seed
    key, rng = random.split(random.PRNGKey(seed))
    a = random.normal(key, [1, 2])

    n_nodes = 64
    model, model_0, test_model= PINN(n_nodes), DNN(n_nodes), TEST(n_nodes)
    params = model.init(key, a)
    num_params, format_params_fn = get_params_format_fn(params)
    params = jax.flatten_util.ravel_pytree(params)[0]
    params_0 = params

    # 设置批量大小
    BS_ALL = 8192
    n_all, n_bic = len(data_X), len(data_X_BIC)

    # 定义损失函数
    def initial_loss(params, inputs, labels, params_0):
        pred = model.apply(format_params_fn(params), inputs)
        u, u_xx, pde, pbc, nbc, ic, bc = jnp.split(pred, 7, axis=1)
        pde_loss = jnp.mean(jnp.square(pde))
        ic_e = (labels - u) * ic
        ic_loss = jnp.sum(jnp.square(ic_e)) / ic.sum()
        pbc_e, nbc_e = pbc * bc, nbc * bc
        bc_loss = jnp.sum(jnp.square(pbc_e)) / bc.sum() + jnp.sum(jnp.square(nbc_e)) / bc.sum()
        return pde_loss + weight_ic * ic_loss + weight_bc * bc_loss
    
    def eval_loss(params, inputs, labels, params_0):
        pred = model.apply(format_params_fn(params), inputs)
        u, u_xx, pde, pbc, nbc, ic, bc = jnp.split(pred, 7, axis=1)
        pred_0 = model_0.apply(format_params_fn(params_0), inputs)
        u_0, u0_xx = jnp.split(pred_0, 2, axis=1)
        if (ER > 0):
            pde = pde + (u - u_0) / ER - v2*(u_xx - u0_xx) / ER_xx
        pde_loss = jnp.mean(jnp.square(pde))
        ic_e = (labels - u) * ic
        ic_loss = jnp.sum(jnp.square(ic_e)) / ic.sum()
        pbc_e, nbc_e = pbc * bc, nbc * bc
        bc_loss = jnp.sum(jnp.square(pbc_e)) / bc.sum() + jnp.sum(jnp.square(nbc_e)) / bc.sum()
        return pde_loss + weight_ic * ic_loss + weight_bc * bc_loss
    
    def compute_error(params):
        u = test_model.apply(format_params_fn(params), data_X)
        rl2 = jnp.linalg.norm(data_Y - u) / jnp.linalg.norm(data_Y)
        return rl2
    
    log_every = 50
    logs = {"iters": [], "rl2": [], "loss": []}
    def callback(it, p, state, cur_loss, _logs=logs):
        if it % log_every == 0 or it == 1:
            rl2 = compute_error(p)
            _logs["iters"].append(it)
            _logs["rl2"].append(float(rl2))
            _logs["loss"].append(float(cur_loss))
        if it % log_every == 0 or it == 1:
            print(
                f"  iter {it:5d} | loss {cur_loss:.6e} | rL2 {float(rl2):.6e}"
            )
        return False

    # 配置优化器
    solver = SSBroyden(rtol=1e-6, atol=1e-6, search=Zoom(initial_guess_strategy="increase"))
    
    # 执行优化
    idx = np.random.permutation(n_all)[:BS_ALL]
    train_X = data_X[idx, :]
    train_Y = data_Y[idx, :]
    train_X = jnp.concatenate([train_X, data_X_BIC], axis=0)
    train_Y = jnp.concatenate([train_Y, data_Y_BIC], axis=0)
    res = run_optimization(
        solver, 
        lambda p_curr: initial_loss(p_curr, train_X, train_Y, params_0),
        params,
        maxiter=max_iter,
        callback=callback,
        verbose=True
    )

    # 评估最终结果
    u = test_model.apply(format_params_fn(res.params), data_X)
    rl2 = jnp.linalg.norm(data_Y - u) / jnp.linalg.norm(data_Y)
    
    print(f'[v2={v2:.1e}]: RL2 = {rl2:.2e}')
    return format_params_fn(res.params)

In [8]:
seed = 10
params = main(ER=0.45, ER_xx=1.5, weight_ic=100, weight_bc=100, max_lr=2e-3, exponent=1.2, max_iter=5000, seed=seed, gpu=0)

D:\ssbroyden_optimistix\optimistix\optimistix\_misc.py:72: UserWarning: Explicitly requested dtype float64 requested in zeros is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  fn = lambda x: jnp.zeros(x.shape, x.dtype)


Compiling solver...
Compilation done.
  iter     1 | loss 2.124519e+02 | rL2 1.396909e+00
  iter    50 | loss 1.176753e+00 | rL2 9.474642e-01
  iter   100 | loss 8.541361e-01 | rL2 7.509426e-01
Compiling solver...
Compilation done.
  iter     1 | loss 9.426094e-01 | rL2 7.509426e-01
  iter    50 | loss 1.081815e-01 | rL2 6.477200e-01
  iter   100 | loss 1.288744e-02 | rL2 6.022913e-01
Compiling solver...
Compilation done.
  iter     1 | loss 3.013971e-01 | rL2 6.022913e-01
  iter    50 | loss 2.523234e-02 | rL2 6.285247e-01
  iter   100 | loss 9.440948e-03 | rL2 5.437084e-01
Compiling solver...
Compilation done.
  iter     1 | loss 6.832024e-02 | rL2 5.437084e-01
  iter    50 | loss 1.487773e-02 | rL2 5.795527e-01
  iter   100 | loss 7.935325e-03 | rL2 4.737950e-01
Compiling solver...
Compilation done.
  iter     1 | loss 4.892226e-02 | rL2 4.737950e-01
  iter    50 | loss 1.242177e-02 | rL2 4.882586e-01
  iter   100 | loss 6.958383e-03 | rL2 4.231173e-01
Compiling solver...
Compilatio

KeyboardInterrupt: 